<a href="https://colab.research.google.com/github/atharva07-SYS/diabetespredictor/blob/main/Diabetes_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import files

uploaded = files.upload()


Saving pima-indians-diabetes[1].csv to pima-indians-diabetes[1].csv


In [2]:
import pandas as pd

df = pd.read_csv('pima-indians-diabetes[1].csv')

print(df.head())

print(df.shape)
print(df.info())

   6  148  72  35    0  33.6  0.627  50  1
0  1   85  66  29    0  26.6  0.351  31  0
1  8  183  64   0    0  23.3  0.672  32  1
2  1   89  66  23   94  28.1  0.167  21  0
3  0  137  40  35  168  43.1  2.288  33  1
4  5  116  74   0    0  25.6  0.201  30  0
(767, 9)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 767 entries, 0 to 766
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   6       767 non-null    int64  
 1   148     767 non-null    int64  
 2   72      767 non-null    int64  
 3   35      767 non-null    int64  
 4   0       767 non-null    int64  
 5   33.6    767 non-null    float64
 6   0.627   767 non-null    float64
 7   50      767 non-null    int64  
 8   1       767 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB
None


In [3]:
import pandas as pd
cols = ['Pregnancies','Glucose','BloodPressure','SkinThickness','Insulin','BMI','DiabetesPedigreeFunction','Age','Outcome']
df = pd.read_csv('pima-indians-diabetes[1].csv',header=None,names=cols)
print(df.shape)
print(df.head())
print(df.describe())

(768, 9)
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  
       Pregnancies     Glucose  BloodPressure  SkinThickness     Insulin  \
count   768.000000  768.000000     768.000000     768.000000  768.000000   
mean      3.845052  120.894531      69.105469      20.536458   79.799479   
std       3.369578   31.972618      19.355807      15.952218  115.

In [4]:
import numpy as np

zero_cols = ['Glucose','BloodPressure','SkinThickness','Insulin','BMI']

print((df[zero_cols] == 0).sum())

df[zero_cols] = df[zero_cols].replace(0,np.nan)

print(df.isnull().sum())

Glucose            5
BloodPressure     35
SkinThickness    227
Insulin          374
BMI               11
dtype: int64
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

#separate features(X) from the label(y)

X = df.drop("Outcome",axis =1)
y = df["Outcome"]
#80% train 20% test

X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2,random_state=42,stratify=y)

print(X_train.shape,X_test.shape)
print("Diabetic share in train:",y_train.mean())
print("Diabetic share in test:",y_test.mean())

#learn medians from Train only , then fill gaps in both
imputer = SimpleImputer(strategy='median')
X_train_imp = imputer.fit_transform(X_train)
X_test_imp = imputer.transform(X_test)

print("Medians learned:",imputer.statistics_)
print("NaNs left in train:",np.isnan(X_train_imp).sum())
print("NaNs left is test",np.isnan(X_test_imp).sum())

(614, 8) (154, 8)
Diabetic share in train: 0.3485342019543974
Diabetic share in test: 0.35064935064935066
Medians learned: [  3.     117.      72.      29.     125.      32.4      0.3825  29.    ]
NaNs left in train: 0
NaNs left is test 0


In [6]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

# Scale: learn from train only
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train_imp)
X_test_sc = scaler.transform(X_test_imp)

# Baseline: always predicts the most common class
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train_sc, y_train)
print("Baseline accuracy:", accuracy_score(y_test, dummy.predict(X_test_sc)))

# Real model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_sc, y_train)
print("Model accuracy:   ", accuracy_score(y_test, model.predict(X_test_sc)))

Baseline accuracy: 0.6493506493506493
Model accuracy:    0.7077922077922078


In [7]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test_sc)

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred, target_names=["Not diabetic", "Diabetic"]))

[[82 18]
 [27 27]]
              precision    recall  f1-score   support

Not diabetic       0.75      0.82      0.78       100
    Diabetic       0.60      0.50      0.55        54

    accuracy                           0.71       154
   macro avg       0.68      0.66      0.67       154
weighted avg       0.70      0.71      0.70       154



In [8]:
probs = model.predict_proba(X_test_sc)[:, 1]   # probability of "diabetic"

for t in [0.5, 0.4, 0.3, 0.2]:
    preds = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    recall = tp / (tp + fn)
    precision = tp / (tp + fp)
    print(f"threshold {t}: caught {tp}/54, false alarms {fp}, "
          f"recall {recall:.2f}, precision {precision:.2f}")

threshold 0.5: caught 27/54, false alarms 18, recall 0.50, precision 0.60
threshold 0.4: caught 35/54, false alarms 22, recall 0.65, precision 0.61
threshold 0.3: caught 43/54, false alarms 29, recall 0.80, precision 0.60
threshold 0.2: caught 48/54, false alarms 39, recall 0.89, precision 0.55


In [9]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf.fit(X_train_imp, y_train)

rf_probs = rf.predict_proba(X_test_imp)[:, 1]

print("LogReg AUC:", roc_auc_score(y_test, probs))
print("RF AUC:    ", roc_auc_score(y_test, rf_probs))

print(classification_report(y_test, rf.predict(X_test_imp),
                            target_names=["Not diabetic", "Diabetic"]))

LogReg AUC: 0.812962962962963
RF AUC:     0.8168518518518517
              precision    recall  f1-score   support

Not diabetic       0.78      0.84      0.81       100
    Diabetic       0.65      0.56      0.60        54

    accuracy                           0.74       154
   macro avg       0.71      0.70      0.70       154
weighted avg       0.73      0.74      0.73       154



In [10]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score

lr_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])

rf_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clf", RandomForestClassifier(n_estimators=300, random_state=42,
                                   class_weight="balanced")),
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, pipe in [("LogReg", lr_pipe), ("RandomForest", rf_pipe)]:
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")
    print(name, scores.round(3), "| mean", scores.mean().round(3),
          "| std", scores.std().round(3))

LogReg [0.867 0.84  0.851 0.849 0.811] | mean 0.843 | std 0.019
RandomForest [0.849 0.809 0.864 0.789 0.806] | mean 0.823 | std 0.028


In [11]:
from sklearn.model_selection import cross_val_predict
import numpy as np

oof_probs = cross_val_predict(lr_pipe, X_train, y_train, cv=cv,
                              method="predict_proba")[:, 1]

n_pos = y_train.sum()
for t in [0.5, 0.45, 0.4, 0.35, 0.3, 0.25, 0.2]:
    preds = (oof_probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_train, preds).ravel()
    print(f"threshold {t}: caught {tp}/{n_pos}, false alarms {fp}, "
          f"recall {tp/(tp+fn):.2f}, precision {tp/(tp+fp):.2f}")

threshold 0.5: caught 123/214, false alarms 39, recall 0.57, precision 0.76
threshold 0.45: caught 136/214, false alarms 53, recall 0.64, precision 0.72
threshold 0.4: caught 146/214, false alarms 66, recall 0.68, precision 0.69
threshold 0.35: caught 152/214, false alarms 82, recall 0.71, precision 0.65
threshold 0.3: caught 167/214, false alarms 117, recall 0.78, precision 0.59
threshold 0.25: caught 179/214, false alarms 141, recall 0.84, precision 0.56
threshold 0.2: caught 191/214, false alarms 177, recall 0.89, precision 0.52


In [12]:
THRESHOLD = 0.4

lr_pipe.fit(X_train, y_train)
test_probs = lr_pipe.predict_proba(X_test)[:, 1]
test_preds = (test_probs >= THRESHOLD).astype(int)

print("Test AUC:", round(roc_auc_score(y_test, test_probs), 3))
print(confusion_matrix(y_test, test_preds))
print(classification_report(y_test, test_preds,
                            target_names=["Not diabetic", "Diabetic"]))

Test AUC: 0.813
[[78 22]
 [19 35]]
              precision    recall  f1-score   support

Not diabetic       0.80      0.78      0.79       100
    Diabetic       0.61      0.65      0.63        54

    accuracy                           0.73       154
   macro avg       0.71      0.71      0.71       154
weighted avg       0.74      0.73      0.74       154



In [13]:
import joblib

# Refit on ALL data (evaluation is done)
final_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("sc", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000)),
])
final_pipe.fit(X, y)

joblib.dump(final_pipe, "diabetes_model.joblib")


def predict_patient(model, patient, threshold=0.4):
    row = pd.DataFrame([patient], columns=X.columns)
    # 0 means "not measured" in these columns
    row[zero_cols] = row[zero_cols].replace(0, np.nan)
    prob = model.predict_proba(row)[0, 1]
    return prob, int(prob >= threshold)


loaded = joblib.load("diabetes_model.joblib")

patient = {"Pregnancies": 2, "Glucose": 90, "BloodPressure": 80,
           "SkinThickness": 0, "Insulin": 0, "BMI": 34.0,
           "DiabetesPedigreeFunction": 0.5, "Age": 45}

prob, label = predict_patient(loaded, patient)
print(f"Probability of diabetes: {prob:.2f} -> {'Diabetic' if label else 'Not diabetic'}")

Probability of diabetes: 0.12 -> Not diabetic


In [14]:
coefs = pd.Series(final_pipe.named_steps["clf"].coef_[0], index=X.columns)
print(coefs.sort_values(ascending=False).round(2))

Glucose                     1.13
BMI                         0.64
Pregnancies                 0.41
DiabetesPedigreeFunction    0.29
Age                         0.15
SkinThickness               0.04
Insulin                    -0.09
BloodPressure              -0.11
dtype: float64
